In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

# Overview
- FER2013 dataset (face images + facial expression)
  + Goal: classifying facial exprression
  + Input: a 48x48 image
  + Output: facial expression (angry, disgust, fear, happy, neutral, sad, and surprise)
- Defining an CNN+MLP
- Training and evaluating an CNN+MLP

# Import libraries

In [ ]:
# Load the original NumPy/CSV data in channels-first format.
def to_tensor_images(array):
    array = np.asarray(array, dtype='float32')
    if array.ndim == 4 and array.shape[-1] <= 4: array = np.transpose(array, (0, 3, 1, 2))
    return torch.from_numpy(array).to(device)

# Set x_train, y_train, x_test and y_test in the next cell using your dataset files.
print('Expected tensors: x=(N,C,H,W), y=(N,) class indices or one-hot labels')

In [ ]:
class ImageCNN(nn.Module):
    def __init__(self, in_channels=1, num_classes=2):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2), nn.Dropout(0.2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2), nn.Dropout(0.2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d((4, 4)))
        self.classifier = nn.Sequential(nn.Flatten(), nn.Linear(128 * 4 * 4, 256), nn.ReLU(), nn.Dropout(0.2), nn.Linear(256, num_classes))
    def forward(self, x): return self.classifier(self.features(x))

# Create the model after loading tensors; labels may be one-hot or class indices.
model = ImageCNN(in_channels=x_train.shape[1], num_classes=num_classes).to(device)
print(model)

# (1) Data

### Download data at https://drive.google.com/drive/folders/0BxINLo5jshCRbXpRcnl2SjJkdlk?resourcekey=0-7qr7Ro7hgkmFknsN1yParQ&usp=sharing
This source code and npy files MUST be in the same location

In [ ]:
# PyTorch training replacement for model.fit / fit_generator.
def fit_model(model, x_train, y_train, x_test, y_test, epochs=30, batch_size=128, lr=1e-4):
    y_train = y_train.argmax(1) if y_train.ndim > 1 else y_train.long()
    y_test = y_test.argmax(1) if y_test.ndim > 1 else y_test.long()
    train_loader = DataLoader(TensorDataset(x_train, y_train), batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(TensorDataset(x_test, y_test), batch_size=batch_size)
    criterion = nn.CrossEntropyLoss(); optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    for epoch in range(epochs):
        model.train()
        for xb, yb in train_loader:
            optimizer.zero_grad(); loss = criterion(model(xb), yb); loss.backward(); optimizer.step()
        model.eval()
        with torch.no_grad():
            logits = torch.cat([model(xb) for xb, _ in test_loader])
            accuracy = (logits.argmax(1) == y_test).float().mean().item()
        print(f'Epoch {epoch + 1}/{epochs}: val_accuracy={accuracy:.4f}')
    return model

In [ ]:
# Inference replacement for model.predict / model.evaluate.
model.eval()
with torch.no_grad():
    logits = model(x_test[:1]); prediction = logits.argmax(1).item()
print('predicted class:', prediction)

# (2) Declare model

# (3) Train defined model
- Note that the training history including loss and accuracy will be save in 'history' variable
- In case your system runs out of memory (OOM), try to decrease batch size

### Training time
- Training time with GTX 1080 is about 6.1120 minutes for 15 epochs
- Training time with Core i7 is around 302.6 minutes (~5.04 hours) for 15 epochs

# (4) Evaluate trained model